In [0]:
from pyspark.sql.functions import col, sum as _sum, countDistinct, round

# 1. Credenciais
storage_account_name = "datalakeecommerce123"
storage_account_access_key = "SUA_CHAVE_AQUI_NO_DATABRICKS_SECRETS"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_access_key
)

# 2. Caminhos dos Containers
path_silver = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/ecommerce"
path_gold_faturamento = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/faturamento_pais"
path_gold_clientes = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/kpi_clientes"

# 3. Ler dados da Camada Silver
df_silver = spark.read.format("delta").load(path_silver)

# Calcular a coluna de Faturamento Total por item (Quantidade * Preço Unitário)
df_silver_calc = df_silver.withColumn("Faturamento", col("Quantity") * col("UnitPrice"))

# --- KPI 1: Faturamento e Vendas por País ---
df_gold_faturamento = (df_silver_calc
    .groupBy("Country")
    .agg(
        round(_sum("Faturamento"), 2).alias("FaturamentoTotal"),
        countDistinct("InvoiceNo").alias("TotalPedidos"),
        countDistinct("CustomerID").alias("TotalClientes")
    )
    .orderBy(col("FaturamentoTotal").desc())
)

# --- KPI 2: Resumo de Compras por Cliente ---
df_gold_clientes = (df_silver_calc
    .groupBy("CustomerID", "Country")
    .agg(
        round(_sum("Faturamento"), 2).alias("TotalGasto"),
        countDistinct("InvoiceNo").alias("QtdPedidos")
    )
    .orderBy(col("TotalGasto").desc())
)

# 4. Salvar as Agregações na Camada Gold (Delta)
(df_gold_faturamento.write
    .format("delta")
    .mode("overwrite")
    .save(path_gold_faturamento)
)

(df_gold_clientes.write
    .format("delta")
    .mode("overwrite")
    .save(path_gold_clientes)
)

print("--- CAMADA GOLD CONCLUÍDA COM SUCESSO! ---")

--- CAMADA GOLD CONCLUÍDA COM SUCESSO! ---


In [0]:
df_resultado = spark.read.format("delta").load(path_gold_faturamento)
display(df_resultado.limit(10))

Country,FaturamentoTotal,TotalPedidos,TotalClientes
United Kingdom,6747156.15,19857,3950
Netherlands,284661.54,101,9
EIRE,250001.78,319,3
Germany,221509.47,603,95
France,196626.05,458,87
Australia,137009.77,69,9
Switzerland,55739.4,71,21
Spain,54756.03,105,31
Belgium,40910.96,119,25
Sweden,36585.41,46,8
